# MÓDULO 4
## Tema 5. Paralelismo, concurrencia y asincronía en Python 

Este cuaderno compara, con una **misma tarea**, cuatro enfoques:

- Secuencial
- Threading (hilos)
- Multiprocessing (procesos)
- Async/await (asincronía)

La tarea es deliberadamente simple y realista: **crear N ficheros con M líneas** y medir tiempos.


## Índice
- Idea general y cuándo usar cada enfoque
- Preparación y parámetros del experimento
- Implementación (secuencial, threading, multiprocessing, asyncio)
- Comparativa de tiempos (tabla + “ganador”)
- Comprobaciones rápidas (que realmente se han escrito los ficheros)
- Trampas comunes y buenas prácticas (GIL, Windows/Jupyter, tamaño del trabajo)


## Resumen rápido (pros y contras)

- **Secuencial**
  - ✅ Sencillo y predecible; sin sobrecostes.
  - ❌ Usa un solo core; lento en tareas paralelizables.

- **Threading (hilos)** (ideal **I/O-bound (cuello de botella I/O)**: disco/red)
  - ✅ Comparte memoria; reduce latencias de espera.
  - ❌ El **GIL (un bloqueo de Python)** limita CPU-bound

- **Multiprocessing (procesos)** (ideal **CPU-bound (cuello de botella CPU)**)
  - ✅ Usa varios cores reales; evita el GIL.
  - ❌ Coste de arranque + serialización; más RAM.

- **Async/await (asincronía)** (muchas I/O concurrentes)
  - ✅ Escala bien con muchas tareas I/O; un solo hilo/event loop.
  - ❌ Requiere código no bloqueante (o derivar a `to_thread`).

## Ejemplos típicos de uso

| Enfoque | Cuándo usarlo | Ejemplos reales |
|---|---|---|
| **Secuencial** | Tareas simples o dependientes entre sí | Scripts pequeños, procesamiento paso a paso, ETL sencillo |
| **Threading** | Tareas I/O-bound (esperas de disco, red, APIs...) | Descargar archivos, scraping web, leer muchos ficheros, peticiones HTTP |
| **Multiprocessing** | Tareas CPU-bound (mucho cálculo) | Machine Learning, procesamiento de imágenes, simulaciones, cálculo numérico |
| **Async/await** | Muchas conexiones o esperas concurrentes | Servidores web, bots, chats, APIs concurrentes, websockets |


## Conceptos clave

- **CPU-bound**: el cuello de botella es la CPU (cálculo). Suele ganar `multiprocessing`.
- **I/O-bound**: el cuello de botella es esperar a disco/red. Suelen ganar `threading` o `asyncio`.

En este cuaderno hacemos una tarea principalmente **I/O-bound (escritura a disco)** para ver patrones prácticos:
pools, futures, límites de concurrencia, medición y verificación.


In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
from pathlib import Path
import shutil
import time
import os

# ─────────────────────────────────────────────────────────────
# Parámetros del experimento (ajusta según tu máquina)
# ─────────────────────────────────────────────────────────────
# N_FICHEROS: cuántos ficheros creamos
# LINEAS_POR_FICHERO: tamaño de cada fichero (más líneas = más trabajo)
#
# Si el experimento termina "demasiado rápido", sube LINEAS_POR_FICHERO.
# Si tarda demasiado, baja LINEAS_POR_FICHERO o N_FICHEROS.
# ─────────────────────────────────────────────────────────────
N_FICHEROS = 200
LINEAS_POR_FICHERO = 2000

# Hilos y procesos por defecto (puedes cambiarlos)
HILOS = min(8, (os.cpu_count() or 4))
PROCESOS = max((os.cpu_count() or 2) - 1, 1)

# Límite de concurrencia para asyncio (evita abrir demasiados ficheros a la vez)
CONC_ASYNCIO = 50

# Carpeta de salida
BASE = Path("OUT_SIMPLE")

In [ ]:
def limpiar_y_crear(ruta: Path) -> None:
    """Borra la carpeta de salida y la recrea."""
    if ruta.exists():
        shutil.rmtree(ruta)
    ruta.mkdir(parents=True, exist_ok=True)

def construir_bloque(n_lineas: int) -> str:
    """Construye el contenido que escribirá cada fichero (1..n_lineas)."""
    return "".join(f"{i}\n" for i in range(1, n_lineas + 1))

def escribir_bloqueante(ruta: Path, bloque: str) -> None:
    """Escritura bloqueante (sin async): abre y escribe."""
    with ruta.open("w", encoding="utf-8") as f:
        f.write(bloque)

def medir(func, *args) -> float:
    """Mide el tiempo de ejecución de una función."""
    t0 = time.perf_counter()
    func(*args)
    t1 = time.perf_counter()
    return round(t1 - t0, 2)

def contar_ficheros_y_bytes(ruta: Path) -> tuple[int, int]:
    """Devuelve (n_ficheros, bytes_totales) de una carpeta."""
    if not ruta.exists():
        return 0, 0
    n = 0
    b = 0
    for p in ruta.glob("*.txt"):
        n += 1
        b += p.stat().st_size
    return n, b


## Implementación

A continuación se implementan los 4 modos, todos escribiendo exactamente lo mismo para comparar tiempos.

In [ ]:
# ─────────────────────────────────────────────
# 1) Secuencial
# ─────────────────────────────────────────────
def modo_secuencial(nf: int, nl: int) -> None:
    out = BASE / "secuencial"
    limpiar_y_crear(out)

    bloque = construir_bloque(nl)

    for i in range(nf):
        ruta = out / f"file_{i:05d}.txt"
        escribir_bloqueante(ruta, bloque)


In [ ]:
# ─────────────────────────────────────────────
# 2) Threading (hilos)
# ─────────────────────────────────────────────
def modo_threading(nf: int, nl: int, hilos: int) -> None:
    out = BASE / "threading"
    limpiar_y_crear(out)

    bloque = construir_bloque(nl)

    with ThreadPoolExecutor(max_workers=hilos) as pool:
        for i in range(nf):
            ruta = out / f"file_{i:05d}.txt"
            pool.submit(escribir_bloqueante, ruta, bloque)
    # Al salir del contexto, el pool espera a que terminen los trabajos pendientes.


In [ ]:
# ─────────────────────────────────────────────
# 3) Multiprocessing (procesos)
# ─────────────────────────────────────────────
def _worker_mp(args) -> None:
    ruta, bloque = args
    with ruta.open("w", encoding="utf-8") as f:
        f.write(bloque)

def modo_multiprocessing(nf: int, nl: int, procesos: int) -> None:
    out = BASE / "multiprocessing"
    limpiar_y_crear(out)

    bloque = construir_bloque(nl)

    tareas = []
    for i in range(nf):
        ruta = out / f"file_{i:05d}.txt"
        tareas.append((ruta, bloque))

    with Pool(processes=procesos) as p:
        p.map(_worker_mp, tareas)


In [ ]:
# ─────────────────────────────────────────────
# 4) Async/await
# ─────────────────────────────────────────────
async def _async_escribir(ruta: Path, bloque: str, sem: asyncio.Semaphore) -> None:
    async with sem:
        # La escritura es bloqueante; la mandamos a un hilo para no parar el event loop
        await asyncio.to_thread(escribir_bloqueante, ruta, bloque)

async def modo_asyncio_async(nf: int, nl: int, max_conc: int) -> None:
    out = BASE / "asyncio"
    limpiar_y_crear(out)

    bloque = construir_bloque(nl)
    sem = asyncio.Semaphore(max_conc)

    tareas = []
    for i in range(nf):
        ruta = out / f"file_{i:05d}.txt"
        tareas.append(_async_escribir(ruta, bloque, sem))

    await asyncio.gather(*tareas)

def modo_asyncio(nf: int, nl: int, max_conc: int) -> None:
    asyncio.run(modo_asyncio_async(nf, nl, max_conc))


### ¿Qué está haciendo cada enfoque?

- **Secuencial**
  
  Va archivo por archivo.  
  Hasta que no termina de escribir uno, no empieza el siguiente.  
  No hay paralelismo ni concurrencia: todo ocurre en un único flujo de ejecución.

- **Threading (hilos)**
  
  Se crea un grupo de hilos (`ThreadPoolExecutor`) y se reparte el trabajo entre ellos.  
  Mientras un hilo está esperando al disco, otro puede seguir trabajando.
  Hay concurrencia, pero en Python el paralelismo real está limitado por el GIL en tareas CPU-bound.

- **Multiprocessing (procesos)**
  
  Se crean varios procesos independientes (`Pool`) y cada uno escribe sus propios archivos.  
  Cada proceso tiene su propio intérprete de Python y puede usar un core distinto de la CPU.
  Es el enfoque con paralelismo real más potente en Python.

- **Async/await**
  
  Se lanzan muchas tareas asíncronas al mismo tiempo (`asyncio.gather`).  
  El event loop va alternando entre tareas mientras unas esperan.
  No hay paralelismo real por sí solo: normalmente trabaja en un único hilo, aunque aquí se apoya en `to_thread` para no bloquear el event loop.

In [ ]:
# ─────────────────────────────────────────────
# Comparativa: ejecutar todo y comparar
# ─────────────────────────────────────────────
def ejecutar_todos(nf: int, nl: int, hilos: int, procesos: int, conc: int):
    resultados = []

    t = medir(modo_secuencial, nf, nl)
    print(f"Secuencial: {t}s")
    resultados.append(("Secuencial", t))

    t = medir(modo_threading, nf, nl, hilos)
    print(f"Threading ({hilos} hilos): {t}s")
    resultados.append((f"Threading ({hilos})", t))

    t = medir(modo_multiprocessing, nf, nl, procesos)
    print(f"Multiprocessing ({procesos} procesos): {t}s")
    resultados.append((f"Multiprocessing ({procesos})", t))

    t = medir(modo_asyncio, nf, nl, conc)
    print(f"Asyncio (máx {conc} tareas): {t}s")
    resultados.append((f"Asyncio (máx {conc})", t))

    print("\nComparativa")
    print("--------------------------------------------")
    for nombre, tiempo in resultados:
        print(f"{nombre:<25} {tiempo:>7.2f}s")
    ganador = min(resultados, key=lambda x: x[1])
    print("--------------------------------------------")
    print(f"Ganador: {ganador[0]} con {ganador[1]:.2f}s")

    return resultados


In [ ]:
# Ejecuta la comparativa con los parámetros actuales
resultados = ejecutar_todos(
    nf=N_FICHEROS,
    nl=LINEAS_POR_FICHERO,
    hilos=HILOS,
    procesos=PROCESOS,
    conc=CONC_ASYNCIO,
)

# Comprobación rápida: ¿cuántos ficheros y bytes se han generado en cada modo?
print("\nVerificación (n_ficheros, bytes_totales):")
for nombre in ["secuencial", "threading", "multiprocessing", "asyncio"]:
    carpeta = BASE / nombre
    n, b = contar_ficheros_y_bytes(carpeta)
    print(f"{nombre:<15} -> {n:>5} ficheros | {b:>10} bytes")


## Notas importantes (para evitar sustos)

- **Jupyter + Windows**: `multiprocessing` puede dar problemas en notebooks por cómo arranca procesos.
  Si te pasa, ejecuta este mismo código como script `.py` (en terminal) o reduce el uso de multiprocessing en notebook.
- **“Ajusta según tu máquina”**: si no ves diferencia, el trabajo es demasiado pequeño.
  Sube `LINEAS_POR_FICHERO` o `N_FICHEROS` hasta que los tiempos sean medibles.
- **No confundas “más concurrencia” con “más rápido”**: demasiadas tareas simultáneas pueden saturar el disco.
  Por eso `asyncio` tiene un semáforo (`CONC_ASYNCIO`).


## Extra: mini-demo CPU-bound (por qué threading no acelera)

En CPU-bound puro (cálculo), el GIL hace que **threading** no escale bien.
Aquí tienes un micro-benchmark para comparar secuencial vs multiprocessing con cálculo.


In [ ]:
import time
from multiprocessing import Pool

def trabajo_pesado(n: int) -> int:
    return sum(i * i for i in range(n))

items = [200_000] * 16  # ajusta según tu máquina

t0 = time.perf_counter()
out_seq = []
for n in items:
    out_seq.append(trabajo_pesado(n))
t1 = time.perf_counter()
print("Secuencial:", round(t1 - t0, 3), "s")

t0 = time.perf_counter()
with Pool() as pool:
    out_par = pool.map(trabajo_pesado, items)
t1 = time.perf_counter()
print("Multiprocessing:", round(t1 - t0, 3), "s")

print("Mismo resultado:", out_seq == out_par)


## Trampas comunes y buenas prácticas

- Usa **threading** para I/O y **multiprocessing** para CPU.
- Mide siempre con `time.perf_counter()` y repite si hace falta.
- Evita abrir demasiados ficheros a la vez (límites del SO). Usa semáforo en asyncio.
- Si vas a escribir en el **mismo fichero** desde varios workers: necesitas locks o diseño distinto.
- En `multiprocessing`, define workers a nivel de módulo (no anidados) para que se puedan serializar.
